# 2.c) Build Soil Suitability Maps — HWSD v2.01 × GAEZ v5 (Module IV)

Construit depuis zéro (formules confirmées mot pour mot depuis le wiki brut GAEZ v5), sans réutiliser
le code de `1_SVM/2_soil_suitability`, sans dépendre de PyAEZ.

**Cultures** : Wheat (dupliqué en Wheat-spring/Wheat-winter, pas de distinction édaphique dans GAEZ),
Maize, Potato, Rice (table "Wetland Rice" pour rainfed ET irrigué — seule option disponible en irrigué,
et pratique dominante mondialement en riziculture pluviale), Soybean.

**Régimes** : rainfed et irrigué gravitaire. **Niveau d'input : intermediate uniquement** pour la
production (voir justification dans `GAEZ_v5_Pipeline_Documentation.docx`) — sauf Maize/rainfed, calculé
aussi en Low et High, conservé pour la validation dans `2.d)_Validate_SoilSuitability.ipynb`.

**Architecture de calcul — point clé** : SQ1-SQ7/SSR sont calculés **séparément pour chaque composant
du sol** (chaque unité de sol au sein d'un mapping unit HWSD, avec ses propres valeurs complètes,
catégorielles incluses), **puis** les résultats (SSR) sont pondérés par part de surface (`SHARE`) —
pas l'inverse (mélanger les entrées puis calculer une fois). Ceci suit littéralement le wiki : *"The
results of soil unit suitability assessment have been **tabulated by the combination of each crop/
soil unit**/slope class/input level/water supply system"* (Module IV). Ce choix architectural a été
vérifié empiriquement : il améliore la corrélation avec le raster GAEZ v5 officiel de ρ=0.71 à ρ=0.84
(Spearman, Maize intermediate) par rapport à une architecture "entrées mélangées d'abord" — voir
`2.d)_Validate_SoilSuitability.ipynb` et la documentation Word pour le détail complet.

**Décisions de conception** (détaillées dans `GAEZ_v5_Pipeline_Documentation.docx`) :
- Permafrost (GSP) retiré de SQ3 — traité dynamiquement dans le futur Module I climatique.
- SQ3 = formule littérale à 4 termes (pas d'ajout phase/obstacle, évite un double comptage avec RSD).
- Pondération par couche (D1-D7) : troncature à la profondeur RSD de référence par culture/régime
  (remplace l'introuvable "Annex 2" du wiki).
- Tables de notation par attribut : classeur officiel `GAEZ4_Appendices.xlsx` (v4), structure de
  combinaison v5 confirmée.

**Portée de ce notebook** : construction à résolution **native** uniquement (une valeur SSR par unité
de sol HWSD2, pas encore rasterisée). L'agrégation en cartes 0.25°/1° est faite séparément dans
`2.e)_Aggregate_SoilSuitability.ipynb`, et la validation (résolution native) dans
`2.d)_Validate_SoilSuitability.ipynb` — les trois étapes sont indépendantes.


## 1. Configuration

In [1]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
from pathlib import Path
import re
import numpy as np
import pandas as pd
import pyodbc
import openpyxl
import rasterio
from rasterio.windows import Window
import xarray as xr

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb
HWSD_TIF = cu.data_file(CFG, "hwsd_raster")  # data path: see 00_config_and_inventory.ipynb
HWSD_MDB = cu.data_file(CFG, "hwsd_mdb")  # data path: see 00_config_and_inventory.ipynb
APPENDIX_XLSX = BASE / "1_SVM" / "2_soil_suitability_gaez" / "GAEZ_soil_ref" / "appendix_tables" / "GAEZ4_Appendices.xlsx"
OUT_DIR = BASE / "1_SVM" / "2_soil_suitability_gaez" / "soil_suitability_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CROPS_APPENDIX = {'Wheat': 'Wheat', 'Maize': 'Maize', 'White Potato': 'Potato',
                   'Wetland Rice': 'Rice', 'Soybean': 'Soybean'}
CROPS = ['Wheat', 'Maize', 'Potato', 'Rice', 'Soybean']
CROP_NAME_ALIASES = {'Wheat': ['Wheat-spring', 'Wheat-winter'], 'Maize': ['Maize'], 'Potato': ['Potato'],
                      'Rice': ['Rice'], 'Soybean': ['Soybean']}
# REGIMES reduit a rainfed seul (demande utilisatrice, 2026-10-13) : le SVM n'utilise que des
# cultures rf, pas la peine de recalculer 'ir' (double le temps de calcul) tant que ce n'est pas
# necessaire ailleurs. Remettre ['rf', 'ir'] si l'irrigue redevient utile.
REGIMES = ['rf']
LEVELS = ['Low', 'Intermediate', 'High']
RATINGS_6 = [100, 90, 70, 50, 30, 10]
AGGS = [(30, 0.25), (120, 1.0)]  # (block factor from 30arcsec native, output deg)

for p in [HWSD_TIF, HWSD_MDB, APPENDIX_XLSX]:
    print(('OK  ' if p.exists() else 'MISSING '), p)


OK   C:\Users\elisa\OneDrive - Politecnico di Milano\PHD_codes\crop_suitability_code_bdap\0_Input_data\soil\HWSD2.bil
OK   C:\Users\elisa\OneDrive - Politecnico di Milano\PHD_codes\crop_suitability_code_bdap\0_Input_data\soil\HWSD2.mdb
OK   C:\Users\elisa\OneDrive - Politecnico di Milano\PHD_docs_state_of_the_art\CROPS\crops suitability\GAEZ4_Appendices.xlsx


## 2. Extraction des tables de notation (`GAEZ4_Appendices.xlsx`), 3 niveaux d'input

Parseur générique par position d'en-tête (pas de largeur de bloc supposée fixe). Note : les feuilles
`A6-3.5`/`A6-3.6` (phases, rainfed Intermediate/Low) utilisent des underscores dans les noms de culture
("Wetland_Rice") au lieu d'espaces partout ailleurs — normalisé explicitement.

In [2]:
wb = openpyxl.load_workbook(APPENDIX_XLSX, data_only=True)


def _cell(ws, r, c):
    return ws.cell(row=r, column=c).value


def _norm_crop_name(name):
    return name.replace('_', ' ').strip() if isinstance(name, str) else name


ATTR_MAP = [
    ('Soil Organic Carbon', 'OC'), ('Soil Reaction', 'pH'), ('Total Exchangeable Bases', 'TEB'),
    ('Cation Exchange Capacity of Clay', 'CECa'), ('Base Saturation', 'BS'),
    ('Rooting/Soil Depth', 'RSD'), ('Coarse Fragments', 'CF'), ('Electric Conductivity', 'EC'),
    ('Exchangeable Sodium', 'ESP'), ('Calcium Carbonate', 'CCB'), ('Calcium Sulphate', 'GYP'),
    ('Vertic Soils Properties', 'VSP'), ('Cation Exchange Capacity Soil', 'CECs'),
    ('Gelic Soil Properties', 'GSP'),
]


def level_matches(label, level):
    return bool(label) and level.lower() in label.lower()


def parse_breakpoint_sheet(sheet_name, crop_col_offset=3):
    ws = wb[sheet_name]
    data_start = None
    for r in range(1, 20):
        v = _cell(ws, r, crop_col_offset)
        if isinstance(v, str) and v.strip() and 'Crops' not in v:
            data_start = r
            break
    label_row = None
    for r in range(data_start - 1, 0, -1):
        v = _cell(ws, r, crop_col_offset + 1)
        if isinstance(v, str) and re.search(r'100\b', v):
            label_row = r
            break
    attr_row, sq_row, input_row = label_row - 3, label_row - 2, label_row - 1
    blocks = []
    col = crop_col_offset + 1
    while col <= ws.max_column:
        attr = _cell(ws, attr_row, col)
        if attr is None:
            col += 1
            continue
        sq = _cell(ws, sq_row, col)
        inp = _cell(ws, input_row, col)
        blocks.append({'col0': col, 'attr': attr.strip(), 'sq': sq.strip() if sq else None,
                        'input': inp.strip() if inp else None})
        col += 6
    out = {}
    r = data_start
    while True:
        name = _cell(ws, r, crop_col_offset)
        if not (isinstance(name, str) and name.strip()):
            break
        name = _norm_crop_name(name)
        if name in CROPS_APPENDIX:
            crop = CROPS_APPENDIX[name]
            out[crop] = []
            for b in blocks:
                vals = [_cell(ws, r, b['col0'] + i) for i in range(6)]
                out[crop].append({'attr': b['attr'], 'sq': b['sq'], 'input': b['input'], 'values': vals})
        r += 1
        if r > data_start + 70:
            break
    return out


def clean_breakpoint_crop(blocks_for_crop, level):
    result = {}
    for b in blocks_for_crop:
        if not level_matches(b['input'], level):
            continue
        key = None
        for sub, k in ATTR_MAP:
            if sub in b['attr']:
                key = k
                break
        if key is None:
            continue
        clean_vals = [None if (v == 999 or v is None) else float(v) for v in b['values']]
        if key == 'pH':
            finite = [v for v in clean_vals if v is not None]
            key = 'pH_low' if (len(finite) >= 2 and finite[0] > finite[-1]) else 'pH_high'
        if key not in result:
            result[key] = {'x': clean_vals, 'rating': list(RATINGS_6)}
    return result


raw_631 = parse_breakpoint_sheet('A6-3.1')
raw_641 = parse_breakpoint_sheet('A6-4.1')
raw_bp = {'rf': raw_631, 'ir': raw_641}
rating_bp = {}
for regime, raw in raw_bp.items():
    rating_bp[regime] = {}
    for level in LEVELS:
        rating_bp[regime][level] = {crop: clean_breakpoint_crop(b, level) for crop, b in raw.items()}
    for crop in CROPS:
        for level in ['Low', 'High']:
            for attr, curve in rating_bp[regime]['Intermediate'][crop].items():
                rating_bp[regime][level][crop].setdefault(attr, curve)

print('Breakpoint OK.', {c: sorted(rating_bp['rf']['Intermediate'][c].keys()) for c in CROPS})


C:\Users\elisa\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\reader\drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


Breakpoint OK. {'Wheat': ['BS', 'CCB', 'CECa', 'CECs', 'CF', 'EC', 'ESP', 'GSP', 'GYP', 'OC', 'RSD', 'TEB', 'VSP', 'pH_high', 'pH_low'], 'Maize': ['BS', 'CCB', 'CECa', 'CECs', 'CF', 'EC', 'ESP', 'GSP', 'GYP', 'OC', 'RSD', 'TEB', 'VSP', 'pH_high', 'pH_low'], 'Potato': ['BS', 'CCB', 'CECa', 'CECs', 'CF', 'EC', 'ESP', 'GSP', 'GYP', 'OC', 'RSD', 'TEB', 'VSP', 'pH_high', 'pH_low'], 'Rice': ['BS', 'CCB', 'CECa', 'CECs', 'CF', 'EC', 'ESP', 'GSP', 'GYP', 'OC', 'RSD', 'TEB', 'VSP', 'pH_high', 'pH_low'], 'Soybean': ['BS', 'CCB', 'CECa', 'CECs', 'CF', 'EC', 'ESP', 'GSP', 'GYP', 'OC', 'RSD', 'TEB', 'VSP', 'pH_high', 'pH_low']}


In [3]:
def find_blocks_by_header(ws, header_row, start_col=3, max_col=90):
    blocks = []
    c = start_col
    while c <= max_col:
        v = _cell(ws, header_row, c)
        if v is not None and str(v).strip():
            col0 = c
            c += 1
            while c <= max_col and _cell(ws, header_row, c) is None:
                c += 1
            blocks.append((col0, c - col0, str(v).strip()))
        else:
            c += 1
    return blocks


def parse_texture(sheet_name, crop_col=3):
    ws = wb[sheet_name]
    data_start = None
    for r in range(1, 15):
        v = _cell(ws, r, crop_col)
        if isinstance(v, str) and v.strip() and 'Crops' not in v:
            data_start = r
            break
    label_row, sq_row, input_row = data_start - 2, data_start - 3, data_start - 4
    blocks = find_blocks_by_header(ws, sq_row, start_col=crop_col + 1)
    out_blocks = []
    for col0, width, _ in blocks:
        sq = _cell(ws, sq_row, col0)
        inp = _cell(ws, input_row, col0)
        classes = [_cell(ws, label_row, col0 + i) for i in range(width)]
        classes = [c.strip() if isinstance(c, str) else c for c in classes]
        out_blocks.append({'col0': col0, 'width': width, 'sq': sq.strip() if sq else None,
                            'input': inp.strip() if inp else None, 'classes': classes})
    result = {}
    r = data_start
    while True:
        name = _cell(ws, r, crop_col)
        if not (isinstance(name, str) and name.strip()):
            break
        name = _norm_crop_name(name)
        if name in CROPS_APPENDIX:
            crop = CROPS_APPENDIX[name]
            result[crop] = []
            for b in out_blocks:
                vals = [_cell(ws, r, b['col0'] + i) for i in range(b['width'])]
                result[crop].append({'sq': b['sq'], 'input': b['input'], 'ratings': dict(zip(b['classes'], vals))})
        r += 1
        if r > data_start + 70:
            break
    return result


def parse_drainage(sheet_name, crop_col=3):
    ws = wb[sheet_name]
    data_start = None
    for r in range(1, 15):
        v = _cell(ws, r, crop_col)
        if isinstance(v, str) and v.strip() and 'Crops' not in v:
            data_start = r
            break
    label_row, group_row, input_row = data_start - 2, data_start - 3, data_start - 4
    blocks = find_blocks_by_header(ws, group_row, start_col=crop_col + 1)
    out_blocks = []
    for col0, width, group in blocks:
        inp = _cell(ws, input_row, col0)
        classes = [_cell(ws, label_row, col0 + i) for i in range(width)]
        classes = [c.strip() if isinstance(c, str) else c for c in classes]
        out_blocks.append({'col0': col0, 'width': width, 'group': group,
                            'input': inp.strip() if inp else None, 'classes': classes})
    result = {}
    r = data_start
    while True:
        name = _cell(ws, r, crop_col)
        if not (isinstance(name, str) and name.strip()):
            break
        name = _norm_crop_name(name)
        if name in CROPS_APPENDIX:
            crop = CROPS_APPENDIX[name]
            result[crop] = []
            for b in out_blocks:
                vals = [_cell(ws, r, b['col0'] + i) for i in range(b['width'])]
                result[crop].append({'group': b['group'], 'input': b['input'], 'ratings': dict(zip(b['classes'], vals))})
        r += 1
        if r > data_start + 70:
            break
    return result


def parse_phase(sheet_name, crop_col):
    ws = wb[sheet_name]
    data_start = None
    for r in range(1, 15):
        v = _cell(ws, r, crop_col - 1)
        if isinstance(v, str) and 'Crops' in v:
            data_start = r + 1
            break
    label_row, sq_row = data_start - 2, data_start - 4
    blocks_sq = find_blocks_by_header(ws, sq_row, start_col=crop_col + 1)
    result = {}
    r = data_start
    while True:
        name = _cell(ws, r, crop_col)
        if not (isinstance(name, str) and name.strip()):
            break
        name = _norm_crop_name(name)
        if name in CROPS_APPENDIX:
            crop = CROPS_APPENDIX[name]
            entries = []
            for col0, width, sqlabel in blocks_sq:
                for i in range(width):
                    phname = _cell(ws, label_row, col0 + i)
                    if not phname:
                        continue
                    val = _cell(ws, r, col0 + i)
                    entries.append({'sq': sqlabel, 'phase': str(phname).strip(), 'rating': val})
            result[crop] = entries
        r += 1
        if r > data_start + 70:
            break
    return result


rating_tex = {'rf': parse_texture('A6-3.2'), 'ir': parse_texture('A6-4.2')}
rating_drn = {'rf': parse_drainage('A6-3.3'), 'ir': parse_drainage('A6-4.3')}
rating_phase = {
    'rf': {'High': parse_phase('A6-3.4', 3), 'Intermediate': parse_phase('A6-3.5', 3), 'Low': parse_phase('A6-3.6', 3)},
    'ir': {'High': parse_phase('A6-4.4', 3), 'Intermediate': parse_phase('A6-4.5', 2)},
}
rating_phase['ir']['Low'] = rating_phase['ir']['Intermediate']  # no Low-input irrigated phase table published
print('Texture/drainage/phase (3 levels, 5 crops) parsed OK.')


Texture/drainage/phase (3 levels, 5 crops) parsed OK.


## 3. Décodage HWSD2 — par composant (pas de mélange des entrées)

Contrairement à une agrégation classique (moyenne pondérée des attributs continus + composant
dominant pour les catégoriels), on garde **chaque composant du sol séparé** (une unité de sol =
une combinaison SMU_ID × SEQUENCE), avec son jeu complet d'attributs (continus ET catégoriels).
La pondération par part de surface (`SHARE`) n'intervient qu'à la toute fin, sur les résultats
SSR — voir section 1 pour la justification (citation littérale du wiki).

In [4]:
conn = pyodbc.connect(r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};DBQ=' + str(HWSD_MDB) + ';')
layers = pd.read_sql('SELECT * FROM HWSD2_LAYERS', conn)


def _domap(table):
    df = pd.read_sql(f'SELECT * FROM {table}', conn)
    return dict(zip(df.iloc[:, 0], df.iloc[:, 1]))


d_phase_map = _domap('D_PHASE')
d_texture_map = _domap('D_TEXTURE_USDA')
d_rootdepth_map = _domap('D_ROOT_DEPTH')

LAYERS_ORDER = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7']
LAYER_DEPTHS = {'D1': (0, 20), 'D2': (20, 40), 'D3': (40, 60), 'D4': (60, 80), 'D5': (80, 100),
                'D6': (100, 150), 'D7': (150, 200)}
CONT_ATTRS = ['SAND', 'SILT', 'CLAY', 'COARSE', 'ORG_CARBON', 'PH_WATER', 'TEB', 'CEC_SOIL',
              'CEC_CLAY', 'BSAT', 'ESP', 'TCARBON_EQ', 'GYPSUM', 'ELEC_COND', 'TEXTURE_USDA']
# WRB_PHASES ajoute (BUGFIX, cf. cellule 5) : code de classification WRB du composant, utilise pour
# detecter les unites "non-sol" (Glaciers/Eau/No-data/Ice sheet) absentes de toute logique existante.
PROFILE_ATTRS = ['ROOT_DEPTH', 'AWC', 'PHASE1', 'PHASE2', 'ADD_PROP', 'ROOTS', 'IL', 'DRAINAGE', 'WRB_PHASES']

layers['DRAINAGE'] = layers['DRAINAGE'].astype(str).str.strip()
layers['LAYER'] = layers['LAYER'].astype(str).str.strip()
layers['WRB_PHASES'] = layers['WRB_PHASES'].astype(str).str.strip()

component_layers, component_profile, component_share = {}, {}, {}
recs = layers[['HWSD2_SMU_ID', 'SEQUENCE', 'LAYER', 'SHARE'] + CONT_ATTRS].to_dict('records')
for rec in recs:
    smu, seq, lyr = int(rec['HWSD2_SMU_ID']), int(rec['SEQUENCE']), rec['LAYER']
    component_layers.setdefault(smu, {}).setdefault(seq, {})[lyr] = {k: rec[k] for k in CONT_ATTRS}
    component_share.setdefault(smu, {})[seq] = rec['SHARE']

d1 = layers[layers['LAYER'] == 'D1'][['HWSD2_SMU_ID', 'SEQUENCE'] + PROFILE_ATTRS].to_dict('records')
for rec in d1:
    smu, seq = int(rec['HWSD2_SMU_ID']), int(rec['SEQUENCE'])
    component_profile.setdefault(smu, {})[seq] = {k: rec[k] for k in PROFILE_ATTRS}

print('components built, n_smu=', len(component_layers))


C:\Users\elisa\AppData\Local\Temp\ipykernel_99644\2141498430.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  layers = pd.read_sql('SELECT * FROM HWSD2_LAYERS', conn)


C:\Users\elisa\AppData\Local\Temp\ipykernel_99644\2141498430.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f'SELECT * FROM {table}', conn)


components built, n_smu= 29538


## 4. Moteur SQ1-SQ7 + SSR (par composant)

In [5]:
ROOTS_CODE_TO_PHASE = {
    0: 'No obstacle  to roots between 0 and 80 cm', 1: 'No obstacle  to roots between 0 and 80 cm',
    2: 'Obstacle to roots between 60 and 80 cm depth', 3: 'Obstacle to roots between 40 and 60 cm depth',
    4: 'Obstacle to roots between 20 and 40 cm depth', 5: 'Obstacle to roots between 0 and 80 cm depth',
    6: 'Obstacle to roots between 0 and 20 cm depth',
}
IL_CODE_TO_PHASE = {
    0: 'No impermeable layer within 150 cm depth', 1: 'No impermeable layer within 150 cm depth',
    2: 'Impermeable layer between 80 and 150 cm depth', 3: 'Impermeable layer between 40 and 80 cm depth',
    4: 'Impermeable layer within 40 cm depth',
}
# BUGFIX #3 (audit formule-par-formule, verification finale demandee par l'utilisatrice) : ces labels
# sont compares aux cles produites par find_blocks_by_header(), qui fait .strip() sur le texte des
# en-tetes Excel -> jamais d'espace de fin. SQ4/SQ5a/SQ6 avaient un espace de fin ici (ex.
# 'SQ4 - Oxygen Availability ') qui ne matchait donc JAMAIS la vraie cle 'SQ4 - Oxygen Availability'
# -> most_limiting_phase_rating() et obstacle_ratings() retournaient TOUJOURS NaN pour SQ4/SQ5/SQ6
# (phase ET obstacle), silencieusement, depuis le debut de l'architecture par composant. Seul SQ7
# (sans espace de fin) fonctionnait par coincidence. Corrige en retirant les espaces de fin.
PHASE_SQ_LABELS = {
    'SQ4': 'SQ4 - Oxygen Availability', 'SQ5a': 'SQ5 - Tolerance for salinity and sodicity',
    'SQ6': 'SQ6 - Tolerance for Lime and Gypsum', 'SQ7': 'SQ7 - Workability',
}
DRAINAGE_ABBREV_TO_NAME = {'VP': 'Very Poor', 'P': 'Poor', 'I': 'Imperfectly', 'MW': 'Moderately well',
                            'W': 'Well', 'SE': 'Somewhat excessive', 'E': 'Excessive'}


def tau(x, curve):
    # BUGFIX (trouve en investiguant un cas glacier alpin, SMU_ID=7005 sous le glacier d'Aletsch) :
    # HWSD2 encode "pas de donnee" avec des sentinels negatifs qui varient par champ/source (-2, -6,
    # -9 confirmes empiriquement sur des composantes WRB_PHASES='GG'/'ND'/'IS') plutot que NULL/NaN
    # de facon uniforme. tau() ne filtrait avant que None/NaN -- un sentinel negatif etait donc
    # interpole comme une vraie valeur physique (ex. "-2% de carbone organique" sur la courbe de
    # notation), produisant une note plausible mais fausse au lieu d'exclure le pixel. Aucune des
    # grandeurs passees a tau() dans ce notebook (texture, pH, CEC, EC, profondeur, etc.) n'est
    # legitimement negative, donc filtrer x<0 comme manquant est sans risque et couvre tous les
    # sentinels HWSD2 d'un coup, sans avoir a les enumerer un par un.
    if x is None or (isinstance(x, (int, float)) and (np.isnan(x) or x < 0)):
        return np.nan
    pts = [(xv, rv) for xv, rv in zip(curve['x'], curve['rating']) if xv is not None]
    if len(pts) == 0:
        return np.nan
    if len(pts) == 1:
        return float(pts[0][1]) if x == pts[0][0] else np.nan
    pts_sorted = sorted(pts, key=lambda p: p[0])
    xp, fp = [p[0] for p in pts_sorted], [p[1] for p in pts_sorted]
    if x <= xp[0]:
        return float(fp[0])
    if x >= xp[-1]:
        return float(fp[-1])
    return float(np.interp(x, xp, fp))


def fSQ(vals):
    v = [x for x in vals if x is not None and not (isinstance(x, float) and np.isnan(x))]
    if not v:
        return np.nan
    if len(v) == 1:
        return v[0]
    j0 = int(np.argmin(v))
    rest = v[:j0] + v[j0 + 1:]
    return v[j0] * (sum(rest) / len(rest)) / 100.0


def fmin(vals):
    v = [x for x in vals if x is not None and not (isinstance(x, float) and np.isnan(x))]
    return min(v) if v else np.nan


def texture_rating(tex_name, crop, regime, level, sq_key):
    blocks = rating_tex[regime][crop]
    want_sq = {'SQ1': 'SQ1- Nutrient Availability', 'SQ2': 'SQ1- Nutrient Availability',
               'SQ3': 'SQ3 - Rooting Conditions', 'SQ7': 'SQ7 - Workability'}[sq_key]
    if sq_key == 'SQ2':
        want_sq = 'SQ2 - Nutrient Retention Capacity'
        for b in blocks:
            if b['sq'] == want_sq and b['input'] == 'High':
                return b['ratings'].get(tex_name, np.nan)
        return texture_rating(tex_name, crop, regime, level, 'SQ1')
    for b in blocks:
        if b['sq'] == want_sq and b['input'] and level.lower() in b['input'].lower():
            return b['ratings'].get(tex_name, np.nan)
    return np.nan


def drainage_rating(drainage_code, crop, regime, level, texture_group):
    if drainage_code is None:
        return np.nan
    dname = DRAINAGE_ABBREV_TO_NAME.get(str(drainage_code).strip())
    if dname is None:
        return np.nan
    for b in rating_drn[regime][crop]:
        if b['group'] == texture_group and b['input'] == level:
            return b['ratings'].get(dname, np.nan)
    return np.nan


def texture_group_from_class(tex_class_id):
    if tex_class_id is None:
        return None
    if tex_class_id in (12, 13):
        return 'Coarse textures'
    if tex_class_id in (1, 2, 3, 8):
        return 'Fine textures'
    return 'Medium textures'


def phase_ratings(phase_name, crop, regime, level):
    entries = rating_phase[regime][level].get(crop, [])
    out = {}
    for e in entries:
        if e['phase'] == phase_name:
            out[e['sq']] = e['rating']
    return out


def most_limiting_phase_rating(prof, crop, regime, level, sq_label):
    ratings = []
    for pcode in [prof.get('PHASE1'), prof.get('PHASE2')]:
        if pcode is None or (isinstance(pcode, float) and np.isnan(pcode)):
            continue
        pname = d_phase_map.get(int(pcode))
        if not pname or pname.strip() == '-':
            continue
        pr = phase_ratings(pname.strip(), crop, regime, level)
        if sq_label in pr and pr[sq_label] not in (None, 999, -999):
            ratings.append(pr[sq_label])
    return fmin(ratings) if ratings else np.nan


def obstacle_ratings(prof, crop, regime, level, sq_label):
    # combine ROOTS/IL (OTR/ISL) via min - mathematiquement correct seulement quand reinjecte dans un
    # autre min (ex. SQ4), via min(a, min(b1,b2)) = min(a,b1,b2). PAS correct si reinjecte dans fSQ
    # (ex. SQ7), voir obstacle_ratings_list ci-dessous.
    out = []
    for code_val, mapping in [(prof.get('ROOTS'), ROOTS_CODE_TO_PHASE), (prof.get('IL'), IL_CODE_TO_PHASE)]:
        if code_val is None or (isinstance(code_val, float) and np.isnan(code_val)):
            continue
        pname = mapping.get(int(code_val))
        if not pname:
            continue
        pr = phase_ratings(pname, crop, regime, level)
        if sq_label in pr and pr[sq_label] not in (None, 999, -999):
            out.append(pr[sq_label])
    return fmin(out) if out else np.nan


def obstacle_ratings_list(prof, crop, regime, level, sq_label):
    # BUGFIX (audit formule-par-formule) : pour SQ7 = fSQ(...), le wiki liste OTR et ISL comme 2 TERMES
    # SEPARES parmi les 8 de la formule (RSD,GRC,TXT,VSP,DRN,SPH,OTR,ISL). Les pre-combiner via min()
    # avant de les passer a fSQ (comme obstacle_ratings ci-dessus, qui est correct pour SQ4 car reinjecte
    # dans un autre min, lui-meme associatif) change le resultat de fSQ, qui lui n'est PAS associatif de
    # la meme facon (min x moyenne-du-reste - un terme "absorbe" dans un min prealable disparait de la
    # moyenne-du-reste au lieu d'y contribuer). Cette fonction retourne donc la liste brute (0, 1 ou 2
    # valeurs) a etaler directement dans les arguments de fSQ pour SQ7, pour coller exactement a la
    # formule a 8 termes du wiki.
    out = []
    for code_val, mapping in [(prof.get('ROOTS'), ROOTS_CODE_TO_PHASE), (prof.get('IL'), IL_CODE_TO_PHASE)]:
        if code_val is None or (isinstance(code_val, float) and np.isnan(code_val)):
            continue
        pname = mapping.get(int(code_val))
        if not pname:
            continue
        pr = phase_ratings(pname, crop, regime, level)
        if sq_label in pr and pr[sq_label] not in (None, 999, -999):
            out.append(pr[sq_label])
    return out


def rootable_depth_s0(crop, regime, level):
    return rating_bp[regime][level][crop]['RSD']['x'][0]


def layer_weights(crop, regime, level):
    rsd = rootable_depth_s0(crop, regime, level)
    weights, total = {}, 0.0
    for lyr in LAYERS_ORDER:
        top, bot = LAYER_DEPTHS[lyr]
        covered = max(0.0, min(bot, rsd) - top)
        weights[lyr] = covered
        total += covered
    if total <= 0:
        return {lyr: (1.0 if lyr == 'D1' else 0.0) for lyr in LAYERS_ORDER}
    return {lyr: w / total for lyr, w in weights.items()}


def compute_sq_for_layer(layer_row, crop, regime, level):
    R = rating_bp[regime][level][crop]
    tex_id = layer_row.get('TEXTURE_USDA')
    tex_name = d_texture_map.get(int(tex_id)) if tex_id is not None and not np.isnan(tex_id) else None
    tex_name = tex_name.strip() if tex_name else None
    oc = tau(layer_row.get('ORG_CARBON'), R['OC'])
    ph = layer_row.get('PH_WATER')
    ph_low = tau(ph, R['pH_low']) if ph is not None else np.nan
    ph_high = tau(ph, R['pH_high']) if ph is not None else np.nan
    ph_rating = fmin([ph_low, ph_high]) if not (np.isnan(ph_low) and np.isnan(ph_high)) else np.nan
    teb = tau(layer_row.get('TEB'), R['TEB'])
    ceca = tau(layer_row.get('CEC_CLAY'), R['CECa'])
    bs = tau(layer_row.get('BSAT'), R['BS'])
    cecs = tau(layer_row.get('CEC_SOIL'), R['CECs'])
    txt_sq1 = texture_rating(tex_name, crop, regime, level, 'SQ1') if tex_name else np.nan
    txt_sq2 = texture_rating(tex_name, crop, regime, level, 'SQ2') if tex_name else np.nan
    return {'OC': oc, 'pH': ph_rating, 'TEB': teb, 'CECa': ceca, 'BS': bs, 'CECs': cecs,
            'TXT1': txt_sq1, 'TXT2': txt_sq2}


def _wmean(pairs):
    pairs = [(v, w) for v, w in pairs if v is not None and not (isinstance(v, float) and np.isnan(v)) and w > 0]
    if not pairs:
        return np.nan
    tot_w = sum(w for _, w in pairs)
    return sum(v * w for v, w in pairs) / tot_w


def compute_sq1_to_sq7_one_component(crop, regime, level, layers_sub, prof):
    R = rating_bp[regime][level][crop]
    w = layer_weights(crop, regime, level)
    sq1_layers, sq2_layers = [], []
    for lyr in LAYERS_ORDER:
        if w[lyr] <= 0 or lyr not in layers_sub:
            continue
        vals = compute_sq_for_layer(layers_sub[lyr], crop, regime, level)
        if lyr == 'D1':
            sq1 = fSQ([vals['TXT1'], vals['OC'], vals['pH'], vals['TEB']])
            sq2 = fSQ([vals['TXT2'], vals['BS'], vals['CECs']])
        else:
            sq1 = fSQ([vals['TXT1'], vals['pH'], vals['TEB']])
            sq2 = fSQ([vals['TXT2'], vals['pH'], vals['BS'], vals['CECa']])
        sq1_layers.append((sq1, w[lyr]))
        sq2_layers.append((sq2, w[lyr]))
    sq1 = _wmean(sq1_layers)
    sq2 = _wmean(sq2_layers)

    rsd_val = prof.get('ROOT_DEPTH')
    rsd_name = d_rootdepth_map.get(int(rsd_val)) if rsd_val is not None and not np.isnan(rsd_val) else None
    rsd_class_depth = {'Deep  (> 100cm)': 100, 'Moderately Deep  (< 100cm)': 75,
                        'Shallow  (< 50cm)': 30, 'Very shallow  (< 10cm)': 5}.get(rsd_name, np.nan)
    rsd_rating = tau(rsd_class_depth, R['RSD'])
    tex_d1 = layers_sub.get('D1')
    tex_name_d1 = None
    if tex_d1 is not None and tex_d1.get('TEXTURE_USDA') is not None and not np.isnan(tex_d1['TEXTURE_USDA']):
        tex_name_d1 = d_texture_map.get(int(tex_d1['TEXTURE_USDA']), '').strip()
    txt_sq3 = texture_rating(tex_name_d1, crop, regime, level, 'SQ3') if tex_name_d1 else np.nan
    grc = tau(tex_d1.get('COARSE') if tex_d1 is not None else None, R['CF'])
    if prof.get('ADD_PROP') == 3:
        vsp = tau(1.0, R['VSP'])
    else:
        vsp = np.nan
    # SQ3 (BUGFIX, audit formule-par-formule contre GAEZ4_Appendices.xlsx) : le wiki ecrit la formule
    # avec 4 termes (TXT,GRC,VSP,GSP), mais la table reelle (A6-3.4 et equivalentes) publie des notes
    # 'SQ3 - Rooting Conditions' dediees et distinctes pour 'Obstacle to roots...' (OTR) et
    # 'Impermeable layer...' (ISL), separees de la courbe continue RSD (qui vient de ROOT_DEPTH, un
    # attribut HWSD different de ROOTS/IL). Ce ne sont donc pas des doublons de RSD : la precedente
    # exclusion ('eviter un double comptage') etait une mauvaise lecture, contredite par la table.
    # Le texte du wiki (ligne 324) dit d'ailleurs explicitement que la profondeur racinaire doit etre
    # ajustee par le minimum incluant 'obstacles to roots or impermeable layers'. OTR/ISL sont donc
    # ajoutes au min, comme pour SQ7 (meme source de notes).
    # GSP (wiki V5 ligne 409) : tau(GSP) = 100 pour les SMU retenus (les unites gelic sont hors
    # evaluation, voir gelic_smu). Ajoute au min comme le wiki l'ecrit ; sans effet numerique ici.
    gsp = 100.0
    sq3_obstacle_list = obstacle_ratings_list(prof, crop, regime, level, 'SQ3 - Rooting Conditions')
    sq3 = rsd_rating * (fmin([txt_sq3, grc, vsp, gsp] + sq3_obstacle_list) / 100.0) if not np.isnan(rsd_rating) else np.nan

    tex_group = texture_group_from_class(int(tex_d1['TEXTURE_USDA'])) if (tex_d1 is not None and
        tex_d1.get('TEXTURE_USDA') is not None and not np.isnan(tex_d1['TEXTURE_USDA'])) else None
    # BUGFIX (trouve en validant composant par composant SQ1-7 contre FAO) : DRAINAGE est un attribut
    # "profil" (constant par composant, stocke dans `prof`/PROFILE_ATTRS), pas un attribut "couche"
    # (CONT_ATTRS, stocke dans `layers_sub`/`tex_d1`). L'ancien code lisait `tex_d1.get('DRAINAGE')`,
    # qui n'existe pas dans ce dict -> toujours None -> SQ4 toujours NaN silencieusement, pour tous
    # les SMU/cultures/regimes/niveaux depuis le debut de l'architecture par composant.
    drainage_code = prof.get('DRAINAGE')
    drg = drainage_rating(drainage_code, crop, regime, level, tex_group) if tex_group else np.nan
    sq4_phase = most_limiting_phase_rating(prof, crop, regime, level, PHASE_SQ_LABELS['SQ4'])
    sq4_obstacle = obstacle_ratings(prof, crop, regime, level, PHASE_SQ_LABELS['SQ4'])
    sq4 = fmin([drg, sq4_phase, sq4_obstacle])

    esp = _wmean([(tau(layers_sub[l].get('ESP'), R['ESP']), w[l]) for l in LAYERS_ORDER if w[l] > 0 and l in layers_sub])
    ec = _wmean([(tau(layers_sub[l].get('ELEC_COND'), R['EC']), w[l]) for l in LAYERS_ORDER if w[l] > 0 and l in layers_sub])
    sq5_phase = most_limiting_phase_rating(prof, crop, regime, level, PHASE_SQ_LABELS['SQ5a'])
    sq5 = fmin([esp, ec, sq5_phase])

    ccb = _wmean([(tau(layers_sub[l].get('TCARBON_EQ'), R['CCB']), w[l]) for l in LAYERS_ORDER if w[l] > 0 and l in layers_sub])
    gyp = _wmean([(tau(layers_sub[l].get('GYPSUM'), R['GYP']), w[l]) for l in LAYERS_ORDER if w[l] > 0 and l in layers_sub])
    sq6_phase = most_limiting_phase_rating(prof, crop, regime, level, PHASE_SQ_LABELS['SQ6'])
    sq6 = fmin([ccb, gyp, sq6_phase])

    txt_sq7 = texture_rating(tex_name_d1, crop, regime, level, 'SQ7') if tex_name_d1 else np.nan
    sq7_phase = most_limiting_phase_rating(prof, crop, regime, level, PHASE_SQ_LABELS['SQ7'])
    # BUGFIX (audit formule-par-formule) : OTR et ISL passes comme 2 arguments separes a fSQ (via
    # obstacle_ratings_list, qui ne les pre-combine pas), conformement aux 8 termes litteraux du wiki
    # (RSD,GRC,TXT,VSP,DRN,SPH,OTR,ISL) - voir obstacle_ratings_list pour le detail de la correction.
    sq7_obstacle_list = obstacle_ratings_list(prof, crop, regime, level, PHASE_SQ_LABELS['SQ7'])
    sq7 = fSQ([rsd_rating, grc, txt_sq7, vsp, drg, sq7_phase] + sq7_obstacle_list)
    return sq1, sq2, sq3, sq4, sq5, sq6, sq7


def ssr_from(sq1, sq2, sq3, sq4, sq5, sq6, sq7, mode):
    f_ssr = fSQ([sq3, sq4, sq5, sq6, sq7])
    if np.isnan(f_ssr):
        return np.nan
    if mode == 'low':
        return sq1 * f_ssr / 100.0 if not np.isnan(sq1) else np.nan
    if mode == 'high':
        return sq2 * f_ssr / 100.0 if not np.isnan(sq2) else np.nan
    if np.isnan(sq1) or np.isnan(sq2):
        return np.nan
    return 0.5 * (sq1 + sq2) * f_ssr / 100.0


print('Engine ready.')


Engine ready.


## 5. Calcul par composant + pondération des résultats

5 cultures × 2 régimes en **intermediate** (production) + Maize/rainfed en **Low et High** en plus
(conservés pour `2.d)_Validate_SoilSuitability.ipynb`).

In [6]:
rows = []
all_smu = list(component_layers.keys())
# Pergelisol (wiki GAEZ v5, Module IV, 'Gelic soil units', ligne 196) : les unites gelic
# (permafrost < 200 cm) sont 'unsuitable for crop growing and excluded from evaluation'.
# -> hors evaluation : aucune ligne SSR pour ces SMU (= NaN en aval). Statique, par SMU HWSD2.
gelic_smu = set(int(x) for x in pd.read_sql('SELECT HWSD2_SMU_ID FROM HWSD2_SMU WHERE ADD_PROP = 2', conn)['HWSD2_SMU_ID'])
print('SMU gelic exclus de l evaluation :', len(gelic_smu))
# BUGFIX (verifie sur le glacier d'Aletsch, HWSD2_SMU_ID=7005) : au-dela du pergelisol (ADD_PROP=2,
# deja gere ci-dessus au niveau SMU), WRB_PHASES identifie des COMPOSANTES "non-sol" (GG=Glaciers,
# WR=Water bodies, ND=No data, IS=Ice sheet -- 253 composantes dans le monde, confirme sur
# HWSD2_LAYERS) dont tous les attributs continus valent un sentinel negatif (-2/-6/-9 selon le
# champ). Avant ce fix, ces composantes passaient par la formule standard (tau() ne filtrait que
# None/NaN) et recevaient une note plausible mais fausse -- ex. SSR=34.6/100 pour le glacier
# d'Aletsch, contre ~0 attendu (confirme avec le raster 1deg existant avant correction). Geree ici
# au niveau COMPOSANT (pas SMU, contrairement au pergelisol) car une SMU peut melanger une part
# glacier/eau avec une part de vrai sol -- SSR=0 pour la part concernee, ponderee par SHARE comme
# les autres, pour que la moyenne ponderee de la SMU soit correctement tiree vers 0 au prorata
# plutot que silencieusement ignoree (ce que ferait NaN dans _wmean).
MISC_WRB = {'GG', 'WR', 'ND', 'IS'}
# BUGFIX #2 (trouve en auditant un ecart Pakistan/Indus vs GAEZ avec l'utilisatrice, 2026-10-13) :
# meme sans code WRB miscellaneous, une composante peut n'avoir AUCUNE donnee reelle (tous les
# attributs continus = sentinel, meme mecanisme que ci-dessus mais sous un autre code WRB "normal",
# ex. 'ARpr' pour SMU 3541 seq=1, 50% de part -- SAND/SILT/CLAY/COARSE tous a -4, TEXTURE_USDA=None).
# Verifie sur toute la base : 100% des 1289 composantes D1 avec TEXTURE_USDA manquant ont aussi
# SAND/SILT/CLAY manquants -- donc texture manquante est un marqueur fiable de "composante vide",
# pas juste d'un trou isole. Sans ce fix, une composante vide recoit quand meme un SQ3 quasi-parfait :
# fmin([texture_manquante=NaN, fragments_manquants=NaN, vsp_manquant=NaN, gsp=100.0]) retombe sur
# la constante gsp codee en dur (toujours valide), et ROOT_DEPTH vaut souvent 1="Deep (>100cm)" meme
# pour une composante sans aucune autre donnee -- combinaison qui donne un score favorable a une
# composante dont on ne sait en realite rien, au lieu de l'exclure. Meme traitement que MISC_WRB :
# SSR=0 pour la part concernee plutot qu'un defaut favorable ou une exclusion qui gonflerait la
# moyenne du reste (meme logique de dilution que le fix precedent).
n_misc_components = 0
n_no_data_components = 0
n = len(all_smu)
for i, smu in enumerate(all_smu):
    if int(smu) in gelic_smu:
        continue
    for crop in CROPS:
        for regime in REGIMES:
            comps = component_layers.get(smu)
            profs = component_profile.get(smu)
            shares = component_share.get(smu)
            if not comps or not profs:
                continue
            # BUGFIX: used to compute Low/High only for Maize/rf (spot-check), Intermediate only
            # for everything else. Needed High (and Low) for all crops to match GAEZ v5's own
            # HRLM/LRLM input levels in 5.a's comparison -- now all 3 levels, every crop/regime.
            need_levels = ['Low', 'Intermediate', 'High']
            acc = {lv: [] for lv in need_levels}
            for seq, layers_sub in comps.items():
                prof = profs.get(seq)
                if prof is None:
                    continue
                share = shares.get(seq, 0)
                if share <= 0:
                    continue
                is_misc = (prof.get('WRB_PHASES') or '').strip() in MISC_WRB
                d1 = layers_sub.get('D1', {})
                tex_d1 = d1.get('TEXTURE_USDA')
                is_no_data = tex_d1 is None or (isinstance(tex_d1, float) and np.isnan(tex_d1))
                if is_misc or is_no_data:
                    if crop == CROPS[0] and regime == REGIMES[0]:
                        if is_misc:
                            n_misc_components += 1
                        else:
                            n_no_data_components += 1
                    for lv in need_levels:
                        acc[lv].append((0.0, share))
                    continue
                for lv in need_levels:
                    sq = compute_sq1_to_sq7_one_component(crop, regime, lv, layers_sub, prof)
                    mode = {'Low': 'low', 'Intermediate': 'int', 'High': 'high'}[lv]
                    acc[lv].append((ssr_from(*sq, mode), share))
            rec = {'SMU_ID': smu, 'crop': crop, 'regime': regime}
            for lv in need_levels:
                key = {'Low': 'SSR_low', 'Intermediate': 'SSR_int', 'High': 'SSR_high'}[lv]
                rec[key] = _wmean(acc[lv])
            rows.append(rec)
    if i % 5000 == 0:
        print(i, '/', n)

print('Composantes non-sol (GG/WR/ND/IS) forcees a SSR=0 :', n_misc_components)
print('Composantes sans aucune donnee reelle (texture manquante) forcees a SSR=0 :', n_no_data_components)
ssr_df = pd.DataFrame(rows)
print(ssr_df.shape)
print(ssr_df.groupby(['crop', 'regime'])['SSR_int'].mean())


SMU gelic exclus de l evaluation : 681
0 / 29538


C:\Users\elisa\AppData\Local\Temp\ipykernel_99644\3429954412.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  gelic_smu = set(int(x) for x in pd.read_sql('SELECT HWSD2_SMU_ID FROM HWSD2_SMU WHERE ADD_PROP = 2', conn)['HWSD2_SMU_ID'])


5000 / 29538


10000 / 29538


15000 / 29538


20000 / 29538


25000 / 29538


Composantes non-sol (GG/WR/ND/IS) forcees a SSR=0 : 253
Composantes sans aucune donnee reelle (texture manquante) forcees a SSR=0 : 944
(144285, 6)
crop     regime
Maize    rf        48.188579
Potato   rf        35.347393
Rice     rf        40.982385
Soybean  rf        34.572311
Wheat    rf        56.353547
Name: SSR_int, dtype: float64


## 6. Sauvegarde — résultat à résolution native (par SMU, pas encore rasterisé)

Ce notebook s'arrête à la résolution native : une valeur SSR par unité de sol (SMU_ID), pas encore
appliquée au raster HWSD2 ni agrégée. L'agrégation (0.25° et 1°) est faite séparément dans
`2.e)_Aggregate_SoilSuitability.ipynb`, pour garder la construction et l'agrégation indépendantes.

In [7]:
ssr_df.to_pickle(OUT_DIR.parent / "_soil_suitability_by_smu.pkl")
# sous-ensemble Maize/rainfed (3 niveaux) pour 2.d)_Validate_SoilSuitability.ipynb
maize_rf_out = ssr_df[(ssr_df['crop'] == 'Maize') & (ssr_df['regime'] == 'rf')]
maize_rf_out.to_pickle(OUT_DIR.parent / "_maize_rf_by_smu.pkl")
print('saved _soil_suitability_by_smu.pkl (', len(ssr_df), 'rows ) and _maize_rf_by_smu.pkl')


saved _soil_suitability_by_smu.pkl ( 144285 rows ) and _maize_rf_by_smu.pkl


## 8. SQ1-SQ7 individuels par SMU (Maize/rainfed, Low & High) — pour validation composant par composant

FAO publie aussi `SQ1`...`SQ7` séparément dans le mapset `SQX` (en plus de `SQ0`, la suitability finale
déjà utilisée dans `2.d)`). On recalcule donc ici, pour Maize/rainfed uniquement (la seule combinaison
validable), les 7 qualités de sol individuelles par SMU (pondérées par composant comme pour SSR), en
Low et High (pour matcher HIM/LIM côté FAO) — afin de comparer `2.d)` notre SQ_k à leur SQ_k, pas
seulement l'agrégat final.

In [8]:
sq_rows = []
crop, regime = 'Maize', 'rf'
for i, smu in enumerate(all_smu):
    comps = component_layers.get(smu)
    profs = component_profile.get(smu)
    shares = component_share.get(smu)
    if not comps or not profs:
        continue
    acc = {'Low': [], 'Intermediate': [], 'High': []}
    for seq, layers_sub in comps.items():
        prof = profs.get(seq)
        if prof is None:
            continue
        share = shares.get(seq, 0)
        if share <= 0:
            continue
        is_misc = (prof.get('WRB_PHASES') or '').strip() in MISC_WRB
        d1 = layers_sub.get('D1', {})
        tex_d1 = d1.get('TEXTURE_USDA')
        is_no_data = tex_d1 is None or (isinstance(tex_d1, float) and np.isnan(tex_d1))
        if is_misc or is_no_data:
            # BUGFIX -- meme exclusion composant "non-sol" / "sans donnee reelle" que la cellule
            # SSR ci-dessus (MISC_WRB et texture manquante), appliquee ici aussi pour que la
            # validation SQ1-7 de 2.d) n'herite pas de la contamination par sentinel ou du biais
            # favorable par defaut sur les composantes Glaciers/Eau/No-data/Ice sheet/vides.
            for lv in ['Low', 'Intermediate', 'High']:
                acc[lv].append((share, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0))
            continue
        for lv in ['Low', 'Intermediate', 'High']:
            sq1, sq2, sq3, sq4, sq5, sq6, sq7 = compute_sq1_to_sq7_one_component(crop, regime, lv, layers_sub, prof)
            acc[lv].append((share, sq1, sq2, sq3, sq4, sq5, sq6, sq7))
    rec = {'SMU_ID': smu}
    for lv, suffix in [('Low', 'low'), ('Intermediate', 'int'), ('High', 'high')]:
        entries = acc[lv]
        if not entries:
            for k in range(1, 8):
                rec[f'SQ{k}_{suffix}'] = np.nan
            continue
        tot_w = sum(e[0] for e in entries)
        for k in range(1, 8):
            pairs = [(e[k], e[0]) for e in entries]
            rec[f'SQ{k}_{suffix}'] = _wmean(pairs) if tot_w > 0 else np.nan
    sq_rows.append(rec)
    if i % 5000 == 0:
        print(i, '/', n)

sq_df = pd.DataFrame(sq_rows)
sq_df.to_pickle(OUT_DIR.parent / "_maize_rf_sq_components_by_smu.pkl")
print('saved _maize_rf_sq_components_by_smu.pkl', sq_df.shape)
sq_df.head()


0 / 29538


5000 / 29538


10000 / 29538


15000 / 29538


20000 / 29538


25000 / 29538


saved _maize_rf_sq_components_by_smu.pkl (29538, 22)


,SMU_ID,SQ1_low,SQ2_low,SQ3_low,SQ4_low,SQ5_low,SQ6_low,SQ7_low,SQ1_int,SQ2_int,...,SQ5_int,SQ6_int,SQ7_int,SQ1_high,SQ2_high,SQ3_high,SQ4_high,SQ5_high,SQ6_high,SQ7_high
0,1666,76.424275,89.921765,33.580,97.0,100.0,98.435294,32.432917,76.424275,89.921765,...,100.0,98.435294,30.537083,78.167467,93.000000,33.580,100.0,100.0,98.460000,22.22875
1,6309,76.146621,87.591890,98.625,99.5,100.0,98.682353,98.195000,76.146621,87.591890,...,100.0,98.682353,98.195000,78.142769,91.012189,98.625,100.0,100.0,98.580000,98.62500
2,1700,69.165438,85.096209,77.270,98.0,100.0,98.613072,52.102500,69.165438,85.096209,...,100.0,98.613072,52.102500,72.123067,87.895000,67.520,100.0,100.0,98.524444,50.95250
3,6310,74.212105,82.381622,99.250,100.0,100.0,97.242572,99.250000,74.212105,82.381622,...,100.0,97.242572,99.250000,76.034738,87.091044,99.250,100.0,100.0,96.514122,99.25000
4,6311,50.878445,65.748052,97.650,95.0,100.0,98.882353,92.650000,50.878445,65.748052,...,100.0,98.882353,92.650000,55.461596,77.940276,97.650,100.0,100.0,98.900000,97.65000


## 7. Suite

- Validation à résolution native (comparaison au raster officiel GAEZ v5, cartes, statistiques) :
  `2.d)_Validate_SoilSuitability.ipynb`.
- Agrégation en rasters 0.25°/1° (production, pour le reste du pipeline) :
  `2.e)_Aggregate_SoilSuitability.ipynb`.
- Prochaine étape du pipeline : Module I (régime climatique).


## 9. Limitation connue — sols hydromorphes (Gleysol/Fluvisol) non traites comme "water-collecting sites"

**Contexte** : en comparant notre SSR_low au raster officiel GAEZ v5 (SQX LIM) restreint a la Scandinavie
(lat 55-70N, lon 5-32E), on observe un biais regional fort et specifique au low input : SQ7 (Workability)
y est sous-estime de -15.8 points en moyenne (vs -5.5 points sur le domaine global) -- voir
`2.d)_Validate_SoilSuitability.ipynb` et l'analyse ad-hoc associee.

**Cause identifiee (pas un bug de formule)** : verifie directement dans `GAEZ4_Appendices.xlsx`, feuille
A6-3.3 "Soil Drainage Ratings" -- pour le mais, drainage "Very Poor" donne DRN=10 en low input contre
DRN=90 en high input (meme sol). C'est intentionnel : GAEZ suppose que le "high input" inclut
implicitement un investissement en drainage artificiel, donc un sol mal draine naturellement est note
severement en low input mais favorablement en high input. Notre code (`drainage_rating()`, utilise dans
`compute_sq1_to_sq7_one_component` pour SQ4 et SQ7) reproduit cette table a la lettre -- ce n'est pas une
erreur d'implementation.

Teste et ecarte : l'agregation par composant au sein d'un SMU (moyenne ponderee par `SHARE` vs composant
dominant seul) ne change la moyenne regionale que de +2.0 points -- bien insuffisant pour expliquer le
biais observe. Ce n'est donc pas non plus un probleme de gestion des SMU multi-composants.

**Vraie cause probable** : le wiki (section "Suitability of water-collecting sites", GAEZ v5 Module IV,
~ligne 676) documente une regle separee pour les sols **Gleysol et Fluvisol naturels (non draines)** : en
**low input**, ils ne passent PAS par la procedure SQ4/SQ7 standard -- ils sont rerouter vers une
classification de tolerance a l'engorgement par groupe de culture (duree de cycle) x classe LGP
(longueur de la periode de croissance), independante des notes SQ1-7. En **high input**, seuls les
Gleysol/Fluvisol draines artificiellement beneficient de la procedure standard (upland). HWSD2 contient
bien les champs necessaires pour identifier un Gleysol (`WRB4`/`FAO90`, confirme sur des SMU scandinaves
reels, ex. `GLeu`/`GLi`, `GLdy`/`GLd`), mais cette regle n'est **pas implementee** dans ce notebook : on
applique partout la procedure SQ1-7 standard, y compris sur les Gleysol/Fluvisol naturels.

**Pourquoi ce n'est pas corrige ici** : la table numerique necessaire ("Suitability make up for short-term
dryland crops in natural water-collecting sites... by LGP class") n'existe que comme image dans le wiki
GitHub (jamais transcrite en texte, absente de `GAEZ4_Appendices.xlsx` -- verifie sur les 36 feuilles du
classeur). Recuperee et transcrite pour verification (groupe "Short-term dry-land crops" uniquement, pas
le groupe du mais) :

| Classe | LGP=0 | 1-29 | 30-59 | 60-89 | 90-119 | 120-149..270-299 (x6) | 300-329..365+ |
|---|---|---|---|---|---|---|---|
| VS | - | - | - | - | - | 33 | - |
| S  | - | - | - | - | 33 | - | - |
| MS | - | - | - | 33 | - | 33 | - |
| mS | - | - | 33 | - | 33 | - | - |
| NS | 100 | 100 | 67 | 67 | 34 | 34 | 100 |

Ce n'est qu'**une table sur ~10** (un groupe de culture par duree de cycle ; les 9 autres, dont celui du
mais, ne sont documentees nulle part dans nos sources). De plus c'est un modele de repartition surfacique
par classe de suitability (pas une note deterministe par pixel comme SQ1-7), et il faudrait une **LGP par
pixel** -- variable du module agro-climatique GAEZ qu'on ne calcule pas dans ce pipeline (on part direct
des bioclimatic variables ERA5 -> SVM). Implementer cette regle correctement serait un sous-module a part
entiere, pas une correction ponctuelle.

**Consequence pratique** : SSR_low (et donc `combined` = climate x soil en low input) est probablement
trop pessimiste sur les sols hydromorphes naturels (Gleysol/Fluvisol non draines), concentres en
Scandinavie/zones boreales -- a garder en tete en interpretant ces resultats, au meme titre que
l'approximation de ponderation par couche racinaire deja actee.


## 10. Limitation connue -- proprietes du sol liees a l'hydrologie, instantane present/historique (pas garanti sous scenario climatique futur)

**Contexte** : en discutant de la limitation Gleysol/Fluvisol ci-dessus (section 9) dans la perspective
de l'usage final de ce pipeline (scenarios AMOC-hosing), une question plus large a ete soulevee :
`WRB_PHASES` (et la classification Gleysol/Fluvisol qui en decoule) n'est pas une mesure en temps reel --
c'est un releve pedologique historique. La morphologie Gleysol/Fluvisol (mouchetures, decoloration par
oxydo-reduction) resulte de decennies de saturation periodique passee. Sous un changement climatique
fort (AMOC-hosing), si la pluviometrie/l'hydrologie locale changent substantiellement, rien ne garantit
que cette classification reste representative de la realite agronomique future a l'horizon du scenario.

**Generalisation importante (pas seulement Gleysol/Fluvisol)** : ce raisonnement s'applique exactement
pareil a l'attribut `DRAINAGE` utilise dans SQ4 pour TOUS les sols, pas seulement ceux marques
Gleysol/Fluvisol -- la classe de drainage HWSD2 est elle aussi un instantane historique. Retirer
uniquement la composante Gleysol serait donc incoherent (on accepterait le risque pour la grande
majorite des sols via DRAINAGE, mais pas pour les quelques % marques Gleysol/Fluvisol specifiquement,
sans justification de pourquoi ceux-la seraient plus suspects). Et "retirer" la penalite ne resout rien
de toute facon : ca remplace une hypothese (pessimiste : "restera humide") par une autre tout aussi non
fondee (optimiste : "sera bien draine"), sans vraie information en plus.

**Decision (demande utilisatrice, 2026-10-13)** : ne pas corriger -- aucune des deux directions
(garder la penalite standard vs. l'assouplir via une heuristique, ex. calendrier MIRCA) n'est mieux
fondee que l'autre sans donnees reelles sur l'hydrologie future. On garde le traitement standard actuel
(DRAINAGE + phase Gleysol/Fluvisol notes comme n'importe quel autre attribut, pas de regle speciale),
et on documente explicitement cette incertitude plutot que de la masquer par un bricolage.

**A retenir pour l'usage en scenario AMOC** : les attributs de sol utilises ici se divisent en deux
categories de fiabilite temporelle differentes --
- **Stables dans le temps** (texture, carbone organique, pH, CEC, salinite/sodicite, calcaire/gypse,
  profondeur de sol) : proprietes physico-chimiques intrinseques du substrat, peu susceptibles de
  changer a l'echelle de temps d'un scenario climatique (decennies).
- **Instantane present/historique, PAS garanti sous climat futur fort** (DRAINAGE, phase
  Gleysol/Fluvisol, et plus largement tout ce qui depend de l'hydrologie locale) : reflete les
  conditions au moment du releve HWSD2, pas une projection. A garder en tete comme une incertitude
  reelle du pipeline, pas une erreur corrigee -- aucune donnee de projection hydro-pedologique future
  n'existe pour la remplacer, et en construire serait un sujet de recherche a part entiere, hors de
  la portee de ce pipeline.
